In [ ]:
import os

import numpy as np
np.random.seed(18)
import matplotlib.pyplot as plt

from scipy.signal import convolve

In [ ]:
#Create folder for storing generated data files
if not os.path.exists("Data_files"):
    warnings.warn("""The folder "Data_files" should already exist. Please, run the "1_Generate_PSFs_with_aberrations" notebook first before proceeding.""")
if not os.path.exists("Data_files/Low_resolution_images"):
    os.mkdir("Data_files/Low_resolution_images")

## Set parameters about the ground truth

In [ ]:
number_of_samples = 100                    #Low-resolution samples per graph point (default 100)

base_size = 50                             #Low-resolution image pixel size (default 50)
scale_factor = 8                           #Targeted model upsampling (default 8)
upscaled_size = base_size*scale_factor

number_of_emitters = 20                    #Emitters per a low-resolution image (default 20)

average_noise = 10                         #Average shot noise value (default 10)
average_intensity = 1e4                    #Average emitter power value (default 1e4)

### Generate Ground truth

Ground truth lives on the upsampled image grid

In [ ]:
ground_truth = np.zeros((number_of_samples, upscaled_size, upscaled_size))

for j in range(number_of_samples):
    for i in range(number_of_emitters):
        emitter_intensity = 0
        while emitter_intensity == 0:
            emitter_intensity = np.random.poisson(average_intensity)
            decrease_factor = np.random.uniform(low = 0.1, high = 1)
             
        ground_truth[j, np.random.randint(0, upscaled_size), np.random.randint(0, upscaled_size)] += emitter_intensity * decrease_factor

print("Shape of ground_truth:", ground_truth.shape)

### Convolve ground truth with the pre-generated aberrated PSFs

In [ ]:
#Load the pre-generated aberrated PSFs
datafile = np.load("Data_files/Aberrated_PSFs/All_stacks_psf_532nm.npz", allow_pickle=True)

all_psf_stacks = datafile["psf_all_stacks"]
aberrations_dictionary = datafile["aberrations_dictionary"].item()

print("Shape of all_psf_stacks:", all_psf_stacks.shape)

In [ ]:
#Allocate variables
all_convolved_stacks = np.zeros([all_psf_stacks.shape[0], all_psf_stacks.shape[1], ground_truth.shape[0], ground_truth.shape[1], ground_truth.shape[2]])

#Cycle over aberration types
for i in range(all_psf_stacks.shape[0]):
    #Cycle over aberration strengths
    for j in range(all_psf_stacks.shape[1]):
        #Get aberrated PSF for the current aberration type and strength
        current_PSF = all_psf_stacks[i,j]
        #Cycle over the ground truth, convolving one-by-one
        for k in range(ground_truth.shape[0]):
            current_convolved_image = convolve(ground_truth[k], current_PSF, mode="same")
            all_convolved_stacks[i,j,k] = current_convolved_image

print("Shape of all_convolved_stacks:", all_convolved_stacks.shape)

In [ ]:
#Show an example of a convolved ground truth
plt.figure(figsize=(5,5))
plt.matshow(all_convolved_stacks[0,0,0], fignum=False)
plt.title("Convolved example")
plt.show()

### Downsample from ground truth image size to low-resolution image size

In [ ]:
#Downsampling by summing individual region
all_convolved_stacks_ds = np.sum(all_convolved_stacks.reshape(all_convolved_stacks.shape[0], all_convolved_stacks.shape[1], all_convolved_stacks.shape[2], 
                                                              base_size, scale_factor, base_size, scale_factor), 
                                 axis=(-3,-1))

print("Shape of all_convolved_stacks_ds:", all_convolved_stacks_ds.shape)

In [ ]:
#Show an example of a downsampled convolved ground truth
plt.figure(figsize=(5,5))
plt.matshow(all_convolved_stacks_ds[0,0,0], fignum=False)
plt.title("Downsampled Convolved example")
plt.show()

### Add noise to the downsampled convolved ground truth

In [ ]:
#Adding Poisson shot noise; random pixel-wise but identical across aberration types and strengths

all_convolved_stacks_ds_noise = all_convolved_stacks_ds + np.random.poisson(average_noise, (number_of_samples, base_size, base_size))[None,None,:,:]

print("Shape of all_convolved_stacks_ds_w_noise:", all_convolved_stacks_ds_noise.shape)

In [ ]:
#Show an example of low-resolution image, i.e., a downsampled convolved ground truth with added noise
plt.figure(figsize=(5,5))
plt.matshow(all_convolved_stacks_ds_noise[0,0,0], fignum=False)
plt.title("Low-resolution image example")
plt.show()

In [ ]:
np.savez("Data_files/Low_resolution_images/Generated_data_with_abberations_532nm.npz", 
         all_stacks = all_convolved_stacks_ds_noise, 
         ground_truth = ground_truth, 
         aberrations_dictionary = aberrations_dictionary)